# Algorithm 20 fixed-time diagnostics

This notebook is a **first-wave diagnostic suite** for Algorithm 20.

It focuses on the minimal experiments most likely to separate:

- denoiser failure,
- reverse-dynamics basin destruction,
- and a witness objective that is too broad or misaligned with RMSE.

Implemented first-wave tests:

1. GT-noised denoiser calibration
2. Denoiser sign / formula sanity
3. GT-fractional oracle PPR
4. q-witness vs q-prior vs GT objective
5. Fixed-time `M` sweep
6. Local reverse sampler ablation
7. Gradient alignment

This notebook deliberately avoids full reverse chains where possible.


In [ ]:
import os
import time
import math
import traceback
import importlib
from collections import Counter
from dataclasses import dataclass, replace
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml

ROOT = Path.cwd().resolve()
if not ((ROOT / 'configs').exists() and (ROOT / 'src').exists()):
    if ((ROOT.parent / 'configs').exists() and (ROOT.parent / 'src').exists()):
        ROOT = ROOT.parent

import kldmPlus.algorithm19_kldm_ppr_diffcsppp as algo19_backend
import kldmPlus.algorithm20_kldm_ppr_q_witness as algo20_backend
algo19_backend = importlib.reload(algo19_backend)
algo20_backend = importlib.reload(algo20_backend)

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction
from kldmPlus.symmetry import build_symmetry_frame_bridge, estimate_semantic_transport_for_reference_order, oracle_spacegroup_from_case
from kldmPlus.utils.time import iter_sampling_times, make_times

Algorithm19State = algo19_backend.Algorithm19State
Algorithm20Config = algo20_backend.Algorithm20Config
q_only_witness_fit = algo20_backend.q_only_witness_fit
ppr_project_step_q_witness = algo20_backend.ppr_project_step_q_witness
ppr_kernel_q_witness = algo20_backend.ppr_kernel_q_witness
witness_torus_sin_loss = algo20_backend.witness_torus_sin_loss
build_oracle_diffcsppp_payload_from_structure = algo19_backend.build_oracle_diffcsppp_payload_from_structure
c_w_ops = algo19_backend.c_w_ops
kldm_clean_fractional_denoiser_Df = algo19_backend.kldm_clean_fractional_denoiser_Df
kldm_ppr_noise_chart = algo19_backend.kldm_ppr_noise_chart
map_model_to_payload_reference_chart = algo19_backend.map_model_to_payload_reference_chart
map_payload_reference_chart_to_model_frame = algo19_backend.map_payload_reference_chart_to_model_frame
torus_rmse = algo19_backend.torus_rmse
wrapdiff = algo19_backend.wrapdiff
wrap01 = algo19_backend.wrap01

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

PROFILE = os.environ.get('KLDM_ALGO20_DIAG_PROFILE', 'laptop').strip().lower()
def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop

def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]

def parse_float_list(text: str) -> list[float]:
    return [float(v.strip()) for v in str(text).split(',') if v.strip()]

SAMPLE_SEED = int(profile_default('KLDM_ALGO20_DIAG_SEED', '2', '2'))
GRAPH_IDS = parse_int_list(profile_default('KLDM_ALGO20_DIAG_GRAPH_IDS', '2', '2'))
CAL_T_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO20_DIAG_CAL_T', '0.05,0.1,0.3,0.5,0.7,0.9', '0.05,0.1,0.3,0.5,0.7,0.9')))
CORE_T_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO20_DIAG_CORE_T', '0.1,0.3,0.5', '0.1,0.3,0.5')))
PROJ_STEPS = int(profile_default('KLDM_ALGO20_DIAG_PROJ_STEPS', '10', '10'))
M_VALUES = tuple(int(v) for v in parse_int_list(profile_default('KLDM_ALGO20_DIAG_M_VALUES', '1,2,4', '1,2,4')))
LAMBDA0_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO20_DIAG_LAMBDA0', '0.1,0.3,1.0', '0.1,0.3,1.0')))
LOCAL_STEPS_VALUES = tuple(int(v) for v in parse_int_list(profile_default('KLDM_ALGO20_DIAG_LOCAL_STEPS', '1,5,10,25', '1,5,10,25')))
ALGO20_Q_ONLY_STEPS = int(profile_default('KLDM_ALGO20_Q_ONLY_STEPS', '100', '100'))
ALGO20_LR = float(profile_default('KLDM_ALGO20_LR', '1e-2', '1e-2'))
ALGO20_LAMBDA_FLOOR = float(profile_default('KLDM_ALGO20_LAMBDA_FLOOR', '1e-6', '1e-6'))
ALGO20_GRAD_CLIP = float(profile_default('KLDM_ALGO20_GRAD_CLIP', '10.0', '10.0'))
ALGO20_SOFT_ANCHOR_TOL = float(profile_default('KLDM_ALGO20_SOFT_ANCHOR_TOL', '1e-5', '1e-5'))
ALGO20_DENOISER_VARIANT = str(profile_default('KLDM_ALGO20_DENOISER_VARIANT', 'minus', 'minus'))
ALGO20_COORDINATE_SCORE_MODE = str(profile_default('KLDM_ALGO20_COORDINATE_SCORE_MODE', 'direct', 'direct'))
Q_PRIOR_WEIGHT = float(profile_default('KLDM_ALGO20_DIAG_Q_PRIOR_WEIGHT', '1.0', '1.0'))
TAU = float(profile_default('KLDM_ALGO20_DIAG_TAU', '0.25', '0.25'))

runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
set_seed(SAMPLE_SEED)

requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()
full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
full_num_graphs = max(len(full_ptr) - 1, 0)
available_graph_ids = [int(graph_id) for graph_id in GRAPH_IDS if 1 <= int(graph_id) <= full_num_graphs]
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in available_graph_ids]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.detach().cpu().tolist()

print(f'profile={PROFILE} graphs={GRAPH_IDS} cal_t={CAL_T_VALUES} core_t={CORE_T_VALUES} proj_steps={PROJ_STEPS} M={M_VALUES} lambda0={LAMBDA0_VALUES}')


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
mps:  False
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
profile=laptop graphs=[2] cal_t=(0.05, 0.1, 0.3, 0.5, 0.7, 0.9) core_t=(0.1, 0.3, 0.5) proj_steps=10 M=(1, 2, 4) lambda0=(0.1, 0.3, 1.0)


In [2]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_l_feature: torch.Tensor
    requested_sg: int

result_tables: dict[str, pd.DataFrame] = {}
payload_cache: dict[int, Any] = {}


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    return dict(sorted(Counter(arr).items()))


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = ((edge_index[0] >= start) & (edge_index[0] < end) & (edge_index[1] >= start) & (edge_index[1] < end))
    return (edge_index[:, mask] - start).detach().clone()


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    for graph_idx0, graph_id in enumerate(graph_ids):
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            composition=composition_counter(g['h']),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_l_feature=g['l'],
            requested_sg=g['sg'],
        ))
    return out

GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    return SimpleNamespace(pos=pos, batch=batch_index, num_atoms=num_atoms, edge_node_index=edge_node_index)


def build_case_structure(case: GraphCase):
    return build_structure_from_sample(case.gt_frac, case.gt_l_feature, case.atomic_numbers, lattice_transform=runner.lattice_transform)


def build_oracle_payload(case: GraphCase):
    if case.graph_id in payload_cache:
        return payload_cache[case.graph_id]
    structure = build_case_structure(case)
    payload = build_oracle_diffcsppp_payload_from_structure(
        standardized_structure=structure,
        requested_spacegroup=oracle_spacegroup_from_case(case),
        tol=1e-2,
    )
    bridge = build_symmetry_frame_bridge(vanilla_structure=structure, standardization='refined', symprec=1e-2, angle_tolerance=5.0)
    tau_ref, assignment_ref, rmse_ref = estimate_semantic_transport_for_reference_order(
        standardized_reference_frac_coords=np.asarray(payload.expanded_frac_coords, dtype=float),
        standardized_reference_atomic_numbers=np.asarray(payload.expanded_atomic_numbers, dtype=int),
        bridge=bridge,
    )
    linear_std_to_model = np.asarray(bridge.standardized_to_vanilla_linear, dtype=float)
    linear_model_to_std = np.linalg.inv(linear_std_to_model)
    payload = replace(
        payload,
        debug_info={
            **(payload.debug_info or {}),
            'model_reference_frac_coords': np.asarray(structure.frac_coords, dtype=float).tolist(),
            'model_to_payload_linear': linear_model_to_std.tolist(),
            'model_to_payload_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'model_to_payload_order': np.asarray(assignment_ref, dtype=int).tolist(),
            'payload_to_model_linear': linear_std_to_model.tolist(),
            'payload_to_model_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'payload_to_model_order': np.asarray(assignment_ref, dtype=int).tolist(),
            'transport_rmse': float(rmse_ref),
        },
    )
    payload_cache[case.graph_id] = payload
    return payload


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    result = evaluate_csp_reconstruction(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    return {'match': bool(result.match), 'valid': bool(result.valid), 'rmse': result.rmse, 'frac_rmse': result.frac_rmse}


def make_algo_state_from_raw(*, f, v, l, atom_types, node_index, edge_node_index, t_graph, t_nodes) -> Algorithm19State:
    return Algorithm19State(
        f=f.detach().clone(),
        v=v.detach().clone(),
        l=l.detach().clone().reshape(-1),
        atom_types=atom_types.detach().clone(),
        node_index=node_index.detach().clone(),
        edge_node_index=edge_node_index.detach().clone(),
        t_graph=t_graph.detach().clone(),
        t_nodes=t_nodes.detach().clone(),
    )


def algo_times(case: GraphCase, t_value: float):
    dtype = case.gt_frac.dtype
    device = case.gt_frac.device
    t_graph = torch.as_tensor([[float(t_value)]], device=device, dtype=dtype)
    t_nodes = torch.full((int(case.gt_frac.shape[0]),), float(t_value), device=device, dtype=dtype)
    return t_graph, t_nodes


def clean_prediction(state: Algorithm19State, *, variant: str = ALGO20_DENOISER_VARIANT, coordinate_score_mode: str = ALGO20_COORDINATE_SCORE_MODE) -> torch.Tensor:
    return kldm_clean_fractional_denoiser_Df(
        model=runner.model,
        f=state.f,
        v=state.v,
        l=state.l,
        atom_types=state.atom_types,
        t_graph=state.t_graph,
        t_nodes=state.t_nodes,
        node_index=state.node_index,
        edge_index=state.edge_node_index,
        variant=variant,
        coordinate_score_mode=coordinate_score_mode,
    )


def sample_gt_noisy_state(case: GraphCase, *, t_value: float, seed: int = SAMPLE_SEED, lattice_override: torch.Tensor | None = None):
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 1000 * int(case.graph_id) + int(round(1000.0 * float(t_value))))
    t_graph, t_nodes = algo_times(case, float(t_value))
    f_t, v_t, eps_v, eps_r, r_t = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=case.gt_frac, index=batch_view.batch)
    lattice = case.gt_l_feature.detach().clone() if lattice_override is None else lattice_override.detach().clone().reshape(-1)
    state = make_algo_state_from_raw(
        f=f_t,
        v=v_t,
        l=lattice,
        atom_types=case.atomic_numbers,
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        t_graph=t_graph,
        t_nodes=t_nodes,
    )
    return state, {'eps_v': eps_v.detach().clone(), 'eps_r': eps_r.detach().clone(), 'r_t': r_t.detach().clone()}


def sample_template_q(case: GraphCase, payload, *, mode: str, seed: int = SAMPLE_SEED, dtype=None, device=None):
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    dtype = case.gt_frac.dtype if dtype is None else dtype
    device = case.gt_frac.device if device is None else device
    if chart.total_dof == 0:
        return torch.empty((0,), device=device, dtype=dtype)
    mode_norm = str(mode).strip().lower()
    if mode_norm in {'crystalformer_oracle', 'oracle_structure', 'oracle'}:
        return torch.as_tensor(chart.q_ref, device=device, dtype=dtype).reshape(-1)
    if mode_norm in {'random_template', 'random'}:
        set_seed(int(seed) + 30000 * int(case.graph_id))
        return torch.rand((int(chart.total_dof),), device=device, dtype=dtype)
    raise ValueError(f'Unsupported q mode {mode!r}')


def build_template_f0_model(case: GraphCase, payload, *, q0: torch.Tensor):
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_eval = torch.remainder(q0.reshape(-1), 1.0) if q0.numel() else q0.reshape(-1)
    z_payload = chart.expand_q(q_eval, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    f0_model = map_payload_reference_chart_to_model_frame(z_payload, payload)
    return f0_model.detach().clone(), z_payload.detach().clone()


def sample_template_noisy_state(case: GraphCase, *, init_mode: str, t_value: float, seed: int = SAMPLE_SEED, lattice_override: torch.Tensor | None = None):
    payload = build_oracle_payload(case)
    q0 = sample_template_q(case, payload, mode=init_mode, seed=seed, dtype=case.gt_frac.dtype, device=case.gt_frac.device)
    f0_model, z0_payload = build_template_f0_model(case, payload, q0=q0)
    batch_view = make_single_graph_batch_view(case, ref_tensor=f0_model)
    set_seed(int(seed) + 70000 * int(case.graph_id) + int(round(1000.0 * float(t_value))) + (0 if init_mode == 'crystalformer_oracle' else 1))
    t_graph, t_nodes = algo_times(case, float(t_value))
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=f0_model, index=batch_view.batch)
    lattice = case.gt_l_feature.detach().clone() if lattice_override is None else lattice_override.detach().clone().reshape(-1)
    state = make_algo_state_from_raw(
        f=f_t,
        v=v_t,
        l=lattice,
        atom_types=case.atomic_numbers,
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        t_graph=t_graph,
        t_nodes=t_nodes,
    )
    return state, {'payload': payload, 'q0': q0.detach().clone(), 'z0_payload': z0_payload.detach().clone()}


def q_distance(a: torch.Tensor, b: torch.Tensor) -> float:
    if a.numel() == 0 and b.numel() == 0:
        return 0.0
    diff = wrapdiff(a.reshape(-1), b.reshape(-1))
    return float(torch.sqrt(diff.square().mean()).detach().item())


def fit_clean_and_stats(clean_f: torch.Tensor, case: GraphCase, payload, *, q_init_mode: str = 'oracle_structure', q_init: torch.Tensor | None = None, seed: int = SAMPLE_SEED):
    z_payload = map_model_to_payload_reference_chart(clean_f, payload)
    fit = q_only_witness_fit(
        z_payload=z_payload,
        payload=payload,
        q_init=q_init,
        q_init_mode=q_init_mode,
        steps=int(ALGO20_Q_ONLY_STEPS),
        lr=float(ALGO20_LR),
        grad_clip=float(ALGO20_GRAD_CLIP),
    )
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_gt = torch.as_tensor(chart.q_ref, device=clean_f.device, dtype=clean_f.dtype).reshape(-1)
    endpoint = evaluate_arrays(case, clean_f, case.gt_l_feature, case.atomic_numbers)
    return {
        'fit': fit,
        'witness_sin': float(fit['witness_sin']),
        'witness_rmse_payload': float(fit['witness_rmse_payload']),
        'q_distance_to_GT': q_distance(fit['q_star'], q_gt),
        'frac_rmse_to_GT': float(endpoint['frac_rmse']),
        'rmse_to_GT': float(endpoint['rmse']) if endpoint['rmse'] is not None else float('nan'),
        'valid': bool(endpoint['valid']),
        'match': bool(endpoint['match']),
    }


def config20(*, M: int, lambda0: float, q_init_mode: str, proj_steps: int) -> Algorithm20Config:
    return Algorithm20Config(
        M=int(M),
        proj_steps=int(proj_steps),
        lr=float(ALGO20_LR),
        lambda_noise=float(lambda0),
        lambda_floor=float(ALGO20_LAMBDA_FLOOR),
        grad_clip=float(ALGO20_GRAD_CLIP),
        anchor_mode='soft',
        denoiser_variant=str(ALGO20_DENOISER_VARIANT),
        coordinate_score_mode=str(ALGO20_COORDINATE_SCORE_MODE),
        soft_anchor_tol=float(ALGO20_SOFT_ANCHOR_TOL),
        q_init_mode=str(q_init_mode),
        q_only_steps=int(ALGO20_Q_ONLY_STEPS),
    )


def safe_display_sorted(df: pd.DataFrame, by: list[str]):
    df = df.copy()
    cols = list(df.columns)
    for key in by:
        if key not in cols:
            df[key] = np.nan
    display(df.sort_values(by).reset_index(drop=True))


loaded_graphs= [2] sg= [4]


In [3]:
def objective_project_step(case: GraphCase, state: Algorithm19State, payload, *, objective: str, lambda0: float, proj_steps: int, q_init_mode: str = 'oracle_structure', q_prior_weight: float = Q_PRIOR_WEIGHT, seed: int = SAMPLE_SEED):
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_gt = torch.as_tensor(chart.q_ref, device=state.f.device, dtype=state.f.dtype).reshape(-1)
    xi_r = torch.zeros_like(state.f, requires_grad=True)
    xi_v = torch.zeros_like(state.v, requires_grad=True)
    needs_q = objective in {'witness', 'witness_qprior'}
    if needs_q:
        q_raw = sample_template_q(case, payload, mode=q_init_mode, seed=seed, dtype=state.f.dtype, device=state.f.device).detach().clone().requires_grad_(True)
        params = [xi_r, xi_v, q_raw]
    else:
        q_raw = None
        params = [xi_r, xi_v]
    optimizer = torch.optim.LBFGS(params, lr=1.0, max_iter=1, history_size=10, line_search_fn='strong_wolfe')
    logs = []

    for step_idx in range(max(int(proj_steps), 1)):
        cache = {}
        def closure():
            optimizer.zero_grad(set_to_none=True)
            f_var, v_var = kldm_ppr_noise_chart(model=runner.model, f_t=state.f, v_t=state.v, xi_r=xi_r, xi_v=xi_v, t_nodes=state.t_nodes, node_index=state.node_index)
            clean = clean_prediction(replace(state, f=f_var, v=v_var))
            if needs_q:
                q = torch.remainder(q_raw, 1.0)
                z_payload = map_model_to_payload_reference_chart(clean, payload)
                z_sym = chart.expand_q(q, device=state.f.device, dtype=state.f.dtype)
                witness = witness_torus_sin_loss(z_payload, z_sym)
                if objective == 'witness_qprior':
                    loss_core = witness + float(q_prior_weight) * wrapdiff(q, q_gt).square().mean()
                else:
                    loss_core = witness
                cache['q_before'] = q.detach()
                cache['z_payload'] = z_payload.detach()
            else:
                loss_core = wrapdiff(clean, case.gt_frac).square().mean()
            lambda_eff, _ = algo20_backend._tdm_lambda_eff(
                model=runner.model,
                t_nodes=state.t_nodes,
                ref_f=state.f,
                ref_v=state.v,
                lambda0=float(lambda0),
                lambda_floor=float(ALGO20_LAMBDA_FLOOR),
            )
            prox = xi_r.square().mean() + xi_v.square().mean()
            loss = loss_core + lambda_eff * prox
            loss.backward()
            cache['loss_core'] = loss_core.detach()
            cache['clean'] = clean.detach()
            return loss
        optimizer.step(closure)
        with torch.no_grad():
            logs.append({
                'step': int(step_idx),
                'loss_core': float(cache['loss_core'].item()),
                'xi_norm': float(torch.sqrt(xi_r.detach().square().mean() + xi_v.detach().square().mean()).item()),
            })

    with torch.no_grad():
        f_star, v_star = kldm_ppr_noise_chart(model=runner.model, f_t=state.f, v_t=state.v, xi_r=xi_r, xi_v=xi_v, t_nodes=state.t_nodes, node_index=state.node_index)
        clean_star = clean_prediction(replace(state, f=f_star, v=v_star))
        q_star = None
        if needs_q:
            q_star = torch.remainder(q_raw, 1.0).detach().clone()
        return {
            'state_star': replace(state, f=f_star.detach().clone(), v=v_star.detach().clone()),
            'clean_star': clean_star.detach().clone(),
            'q_star': q_star,
            'logs': logs,
            'xi_norm': float(torch.sqrt(xi_r.detach().square().mean() + xi_v.detach().square().mean()).item()),
        }


def renoise_clean_to_same_t(state: Algorithm19State, clean_f: torch.Tensor, *, seed: int = SAMPLE_SEED):
    set_seed(int(seed) + 880000 + int(round(float(state.t_nodes[0].item()) * 1000)))
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=state.t_nodes, f0=clean_f, index=state.node_index)
    return replace(state, f=f_t.detach().clone(), v=v_t.detach().clone())


def current_stochastic_step(state: Algorithm19State, *, dt: float):
    preds = runner.model.score_network(t=state.t_graph, pos=state.f, v=state.v, h=state.atom_types, l=state.l.reshape(1,-1), node_index=state.node_index, edge_node_index=state.edge_node_index)
    f2, v2 = runner.model.tdm.reverse_step_predictor(t=state.t_nodes, f_t=state.f, v_t=state.v, pred_v=preds['v'], index=state.node_index, dt=dt)
    preds2 = runner.model.score_network(t=torch.as_tensor([[max(float(state.t_graph.item())-dt, 1e-6)]], device=state.f.device, dtype=state.f.dtype), pos=f2, v=v2, h=state.atom_types, l=state.l.reshape(1,-1), node_index=state.node_index, edge_node_index=state.edge_node_index)
    f3, v3 = runner.model.tdm.reverse_step_corrector(t=torch.full_like(state.t_nodes, max(float(state.t_nodes[0].item())-dt,1e-6)), f_t=f2, v_t=v2, pred_v=preds2['v'], dt=dt, index=state.node_index, tau=TAU)
    return replace(state, f=f3.detach().clone(), v=v3.detach().clone(), t_graph=torch.as_tensor([[max(float(state.t_graph.item())-dt,1e-6)]], device=state.f.device, dtype=state.f.dtype), t_nodes=torch.full_like(state.t_nodes, max(float(state.t_nodes[0].item())-dt,1e-6)))


def no_noise_reverse_step(state: Algorithm19State, *, dt: float):
    preds = runner.model.score_network(t=state.t_graph, pos=state.f, v=state.v, h=state.atom_types, l=state.l.reshape(1,-1), node_index=state.node_index, edge_node_index=state.edge_node_index)
    f2, v2 = runner.model.tdm.reverse_step_predictor(t=state.t_nodes, f_t=state.f, v_t=state.v, pred_v=preds['v'], index=state.node_index, dt=dt)
    return replace(state, f=f2.detach().clone(), v=v2.detach().clone(), t_graph=torch.as_tensor([[max(float(state.t_graph.item())-dt,1e-6)]], device=state.f.device, dtype=state.f.dtype), t_nodes=torch.full_like(state.t_nodes, max(float(state.t_nodes[0].item())-dt,1e-6)))


def flow_ode_like_step(state: Algorithm19State, *, dt: float):
    out_v = runner.model.tdm.reconstruct_full_reverse_velocity_score(
        t=state.t_nodes,
        v_t=state.v,
        pred_v=runner.model.score_network(t=state.t_graph, pos=state.f, v=state.v, h=state.atom_types, l=state.l.reshape(1,-1), node_index=state.node_index, edge_node_index=state.edge_node_index)['v'],
        index=state.node_index,
    )
    dt_internal = torch.as_tensor(runner.model.tdm.T * dt, device=state.v.device, dtype=state.v.dtype)
    exp_dt = torch.exp(dt_internal)
    expm1_dt = torch.expm1(dt_internal)
    score_scale = torch.as_tensor(runner.model.tdm.vel_scale**2, device=state.v.device, dtype=state.v.dtype)
    v_prev = exp_dt * state.v + 1.0 * score_scale * expm1_dt * out_v
    f_prev = runner.model.tdm.wrap_displacements(state.f - dt_internal * v_prev)
    return replace(state, f=f_prev.detach().clone(), v=v_prev.detach().clone(), t_graph=torch.as_tensor([[max(float(state.t_graph.item())-dt,1e-6)]], device=state.f.device, dtype=state.f.dtype), t_nodes=torch.full_like(state.t_nodes, max(float(state.t_nodes[0].item())-dt,1e-6)))


## Test 1 — GT-noised denoiser calibration

**Purpose:** Check whether KLDM denoises back toward GT when we start from a GT-noised state. This is the fastest way to see whether `D_f` is locally trustworthy at each noise level before we blame PPR.


In [4]:
rows = []
print(f'GT-noised calibration graphs={[c.graph_id for c in GRAPH_CASES]} t={CAL_T_VALUES}')
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for t_value in CAL_T_VALUES:
        print(f'  graph={case.graph_id} t={t_value:.2f}')
        try:
            state, _ = sample_gt_noisy_state(case, t_value=float(t_value))
            clean = clean_prediction(state)
            stats = fit_clean_and_stats(clean, case, payload, q_init_mode='oracle_structure')
            rows.append({
                'test': 'algo20_gt_noised_denoiser_calibration',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                't': float(t_value),
                'denoiser_frac_rmse_to_GT': stats['frac_rmse_to_GT'],
                'denoiser_witness_sin': stats['witness_sin'],
                'q_distance_to_GT': stats['q_distance_to_GT'],
                'valid': stats['valid'],
                'match': stats['match'],
                'PASS': bool(stats['frac_rmse_to_GT'] < 0.05 if t_value <= 0.1 else True),
                'status': 'INFO',
            })
        except Exception as exc:
            rows.append(error_row('algo20_gt_noised_denoiser_calibration', case.graph_id, exc, 'gt_noised_denoiser_calibration', space_group=case.requested_sg, t=float(t_value)))

df = dataframe_result('algo20_gt_noised_denoiser_calibration', rows)
safe_display_sorted(df, ['graph','t'])


GT-noised calibration graphs=[2] t=(0.05, 0.1, 0.3, 0.5, 0.7, 0.9)
  graph=2 t=0.05
  graph=2 t=0.10
  graph=2 t=0.30
  graph=2 t=0.50
  graph=2 t=0.70
  graph=2 t=0.90


,test,graph,space_group,t,denoiser_frac_rmse_to_GT,denoiser_witness_sin,q_distance_to_GT,valid,match,PASS,status
0,algo20_gt_noised_denoiser_calibration,2,4,0.05,0.002054,0.000003,0.001986,True,True,True,INFO
1,algo20_gt_noised_denoiser_calibration,2,4,0.10,0.002882,0.000007,0.002765,True,True,True,INFO
2,algo20_gt_noised_denoiser_calibration,2,4,0.30,0.005125,0.000036,0.004750,True,True,True,INFO
3,algo20_gt_noised_denoiser_calibration,2,4,0.50,0.007674,0.000196,0.006245,True,True,True,INFO
4,algo20_gt_noised_denoiser_calibration,2,4,0.70,0.014207,0.001072,0.009645,True,True,True,INFO
5,algo20_gt_noised_denoiser_calibration,2,4,0.90,0.154471,0.127312,0.149554,True,False,True,INFO


## Test 2 — Denoiser sign / formula sanity

**Purpose:** Compare the `minus` and `plus` clean-denoiser conventions on the same GT-noised states. This is a cheap sanity check for sign or formula mistakes in `D_f`.


In [5]:
rows = []
variants = ('minus', 'plus')
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for t_value in CORE_T_VALUES:
        state, _ = sample_gt_noisy_state(case, t_value=float(t_value))
        for variant in variants:
            try:
                clean = clean_prediction(state, variant=variant, coordinate_score_mode='direct')
                stats = fit_clean_and_stats(clean, case, payload, q_init_mode='oracle_structure')
                rows.append({
                    'test': 'algo20_denoiser_variant_sanity',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    't': float(t_value),
                    'variant': variant,
                    'frac_rmse_to_GT': stats['frac_rmse_to_GT'],
                    'witness_sin': stats['witness_sin'],
                    'q_distance_to_GT': stats['q_distance_to_GT'],
                    'PASS': True,
                    'status': 'INFO',
                })
            except Exception as exc:
                rows.append(error_row('algo20_denoiser_variant_sanity', case.graph_id, exc, 'denoiser_variant_sanity', space_group=case.requested_sg, t=float(t_value), variant=variant))

df = dataframe_result('algo20_denoiser_variant_sanity', rows)
safe_display_sorted(df, ['graph','t','variant'])


,test,graph,space_group,t,variant,frac_rmse_to_GT,witness_sin,q_distance_to_GT,PASS,status
0,algo20_denoiser_variant_sanity,2,4,0.1,minus,0.002882,0.000007,0.002765,True,INFO
1,algo20_denoiser_variant_sanity,2,4,0.1,plus,0.010097,0.000340,0.008212,True,INFO
2,algo20_denoiser_variant_sanity,2,4,0.3,minus,0.005125,0.000036,0.004750,True,INFO
3,algo20_denoiser_variant_sanity,2,4,0.3,plus,0.058832,0.017427,0.040675,True,INFO
4,algo20_denoiser_variant_sanity,2,4,0.5,minus,0.007674,0.000196,0.006245,True,INFO
5,algo20_denoiser_variant_sanity,2,4,0.5,plus,0.122363,0.056590,0.100540,True,INFO


## Tests 3–4 — Oracle GT-PPR and objective comparison

**Purpose:** Compare three projection objectives on the same fixed states: pure witness, witness plus q-prior, and an unfair GT-fractional oracle loss. This tells us whether the machinery can move toward GT at all, and whether the current witness objective is too broad.


In [6]:
rows = []
objective_modes = ('witness', 'witness_qprior', 'gt_frac')
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for t_value in CORE_T_VALUES:
        state, _ = sample_gt_noisy_state(case, t_value=float(t_value))
        base_clean = clean_prediction(state)
        base_stats = fit_clean_and_stats(base_clean, case, payload, q_init_mode='oracle_structure')
        for objective in objective_modes:
            for lambda0 in LAMBDA0_VALUES:
                try:
                    project = objective_project_step(case, state, payload, objective=objective, lambda0=float(lambda0), proj_steps=int(PROJ_STEPS))
                    proj_clean = project['clean_star']
                    proj_stats = fit_clean_and_stats(proj_clean, case, payload, q_init_mode='oracle_structure')
                    red_state = renoise_clean_to_same_t(state, proj_clean)
                    red_clean = clean_prediction(red_state)
                    red_stats = fit_clean_and_stats(red_clean, case, payload, q_init_mode='oracle_structure')
                    rows.append({
                        'test': 'algo20_objective_compare',
                        'graph': case.graph_id,
                        'space_group': case.requested_sg,
                        't': float(t_value),
                        'objective': objective,
                        'lambda0': float(lambda0),
                        'witness_before': base_stats['witness_sin'],
                        'witness_after': proj_stats['witness_sin'],
                        'frac_rmse_before': base_stats['frac_rmse_to_GT'],
                        'frac_rmse_after_project': proj_stats['frac_rmse_to_GT'],
                        'frac_rmse_after_redenoise': red_stats['frac_rmse_to_GT'],
                        'q_distance_to_GT_before': base_stats['q_distance_to_GT'],
                        'q_distance_to_GT_after': proj_stats['q_distance_to_GT'],
                        'c_after_redenoise': red_stats['witness_sin'],
                        'xi_norm': project['xi_norm'],
                        'PASS': True,
                        'status': 'INFO',
                    })
                except Exception as exc:
                    rows.append(error_row('algo20_objective_compare', case.graph_id, exc, 'objective_compare', space_group=case.requested_sg, t=float(t_value), objective=objective, lambda0=float(lambda0)))

df = dataframe_result('algo20_objective_compare', rows)
safe_display_sorted(df, ['graph','t','objective','lambda0'])


,test,graph,space_group,t,objective,lambda0,witness_before,witness_after,frac_rmse_before,frac_rmse_after_project,frac_rmse_after_redenoise,q_distance_to_GT_before,q_distance_to_GT_after,c_after_redenoise,xi_norm,PASS,status
0,algo20_objective_compare,2,4,0.1,gt_frac,0.1,0.000007,0.000007,0.002882,0.002882,0.003807,0.002765,0.002765,0.000008,0.000000,True,INFO
1,algo20_objective_compare,2,4,0.1,gt_frac,0.3,0.000007,0.000007,0.002882,0.002882,0.003807,0.002765,0.002765,0.000008,0.000000,True,INFO
2,algo20_objective_compare,2,4,0.1,gt_frac,1.0,0.000007,0.000007,0.002882,0.002882,0.003807,0.002765,0.002765,0.000008,0.000000,True,INFO
3,algo20_objective_compare,2,4,0.1,witness,0.1,0.000007,0.000007,0.002882,0.002882,0.003807,0.002765,0.002765,0.000008,0.000009,True,INFO
4,algo20_objective_compare,2,4,0.1,witness,0.3,0.000007,0.000007,0.002882,0.002882,0.003807,0.002765,0.002765,0.000008,0.000009,True,INFO
5,algo20_objective_compare,2,4,0.1,witness,1.0,0.000007,0.000007,0.002882,0.002882,0.003807,0.002765,0.002765,0.000008,0.000009,True,INFO
6,algo20_objective_compare,2,4,0.1,witness_qprior,0.1,0.000007,0.000007,0.002882,0.002882,0.003807,0.002765,0.002765,0.000008,0.000009,True,INFO
7,algo20_objective_compare,2,4,0.1,witness_qprior,0.3,0.000007,0.000007,0.002882,0.002882,0.003807,0.002765,0.002765,0.000008,0.000009,True,INFO
8,algo20_objective_compare,2,4,0.1,witness_qprior,1.0,0.000007,0.000007,0.002882,0.002882,0.003807,0.002765,0.002765,0.000008,0.000009,True,INFO
9,algo20_objective_compare,2,4,0.3,gt_frac,0.1,0.000036,0.000030,0.005125,0.004034,0.005526,0.004750,0.003643,0.000041,0.029069,True,INFO


## Test 5 — Fixed-time `M` sweep

**Purpose:** Measure whether repeating the PPR kernel at one fixed time helps or harms witness, RMSE, and q-basin stability. This is the lightweight version of the paper's repeated project-renoise idea.


In [7]:
rows = []
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for t_value in CORE_T_VALUES:
        state, _ = sample_gt_noisy_state(case, t_value=float(t_value))
        base_clean = clean_prediction(state)
        base_stats = fit_clean_and_stats(base_clean, case, payload, q_init_mode='oracle_structure')
        for M in M_VALUES:
            try:
                kernel = ppr_kernel_q_witness(
                    state=state,
                    payload=payload,
                    model=runner.model,
                    config=config20(M=int(M), lambda0=0.3, q_init_mode='oracle_structure', proj_steps=int(PROJ_STEPS)),
                    q_init=None,
                )
                clean_after = clean_prediction(kernel.state)
                after_stats = fit_clean_and_stats(clean_after, case, payload, q_init_mode='oracle_structure', q_init=kernel.q_star)
                rows.append({
                    'test': 'algo20_fixed_time_M_sweep',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    't': float(t_value),
                    'M': int(M),
                    'witness_before': base_stats['witness_sin'],
                    'witness_after': after_stats['witness_sin'],
                    'frac_rmse_before': base_stats['frac_rmse_to_GT'],
                    'frac_rmse_after': after_stats['frac_rmse_to_GT'],
                    'q_distance_to_GT': after_stats['q_distance_to_GT'],
                    'valid': after_stats['valid'],
                    'PASS': True,
                    'status': 'INFO',
                })
            except Exception as exc:
                rows.append(error_row('algo20_fixed_time_M_sweep', case.graph_id, exc, 'fixed_time_M_sweep', space_group=case.requested_sg, t=float(t_value), M=int(M)))

df = dataframe_result('algo20_fixed_time_M_sweep', rows)
safe_display_sorted(df, ['graph','t','M'])


,test,graph,space_group,t,M,witness_before,witness_after,frac_rmse_before,frac_rmse_after,q_distance_to_GT,valid,PASS,status
0,algo20_fixed_time_M_sweep,2,4,0.1,1,0.000007,0.000005,0.002882,0.003743,0.003680,True,True,INFO
1,algo20_fixed_time_M_sweep,2,4,0.1,2,0.000007,0.000005,0.002882,0.004031,0.003962,True,True,INFO
2,algo20_fixed_time_M_sweep,2,4,0.1,4,0.000007,0.000006,0.002882,0.004771,0.004705,True,True,INFO
3,algo20_fixed_time_M_sweep,2,4,0.3,1,0.000036,0.000026,0.005125,0.005109,0.004844,True,True,INFO
4,algo20_fixed_time_M_sweep,2,4,0.3,2,0.000036,0.000022,0.005125,0.005550,0.005343,True,True,INFO
5,algo20_fixed_time_M_sweep,2,4,0.3,4,0.000036,0.000032,0.005125,0.005691,0.005395,True,True,INFO
6,algo20_fixed_time_M_sweep,2,4,0.5,1,0.000196,0.000074,0.007674,0.006010,0.005348,True,True,INFO
7,algo20_fixed_time_M_sweep,2,4,0.5,2,0.000196,0.000092,0.007674,0.007615,0.006973,True,True,INFO
8,algo20_fixed_time_M_sweep,2,4,0.5,4,0.000196,0.000221,0.007674,0.010343,0.009200,True,True,INFO


## Test 6 — Local reverse sampler ablation

**Purpose:** See whether a short reverse segment destroys a good noised seed. We compare the current stochastic reverse step to deterministic no-noise and flow-ODE-like ablations.


In [8]:
rows = []
sampler_modes = ('current_stochastic', 'no_noise_reverse', 'flow_ode_like')
step_fns = {
    'current_stochastic': current_stochastic_step,
    'no_noise_reverse': no_noise_reverse_step,
    'flow_ode_like': flow_ode_like_step,
}
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for t_start in (0.3, 0.5, 0.7):
        for n_local_steps in LOCAL_STEPS_VALUES:
            for sampler_mode in sampler_modes:
                try:
                    state, _ = sample_gt_noisy_state(case, t_value=float(t_start))
                    dt = max((float(t_start) - max(float(t_start) - 0.1, 1e-6)) / max(int(n_local_steps), 1), 1e-6)
                    curr = state
                    for _ in range(int(n_local_steps)):
                        curr = step_fns[sampler_mode](curr, dt=dt)
                    clean = clean_prediction(curr)
                    stats = fit_clean_and_stats(clean, case, payload, q_init_mode='oracle_structure')
                    rows.append({
                        'test': 'algo20_local_reverse_ablation',
                        'graph': case.graph_id,
                        'space_group': case.requested_sg,
                        't_start': float(t_start),
                        'n_local_steps': int(n_local_steps),
                        'sampler_mode': sampler_mode,
                        'frac_rmse_to_GT': stats['frac_rmse_to_GT'],
                        'witness_sin': stats['witness_sin'],
                        'q_distance_to_GT': stats['q_distance_to_GT'],
                        'PASS': True,
                        'status': 'INFO',
                    })
                except Exception as exc:
                    rows.append(error_row('algo20_local_reverse_ablation', case.graph_id, exc, 'local_reverse_ablation', space_group=case.requested_sg, t_start=float(t_start), n_local_steps=int(n_local_steps), sampler_mode=sampler_mode))

df = dataframe_result('algo20_local_reverse_ablation', rows)
safe_display_sorted(df, ['graph','t_start','n_local_steps','sampler_mode'])


,test,graph,space_group,t_start,n_local_steps,sampler_mode,frac_rmse_to_GT,witness_sin,q_distance_to_GT,PASS,status
0,algo20_local_reverse_ablation,2,4,0.3,1,current_stochastic,0.004880,0.000026,0.004603,True,INFO
1,algo20_local_reverse_ablation,2,4,0.3,1,flow_ode_like,0.004565,0.000021,0.004326,True,INFO
2,algo20_local_reverse_ablation,2,4,0.3,1,no_noise_reverse,0.004531,0.000018,0.004320,True,INFO
3,algo20_local_reverse_ablation,2,4,0.3,5,current_stochastic,0.004970,0.000011,0.004859,True,INFO
4,algo20_local_reverse_ablation,2,4,0.3,5,flow_ode_like,0.004259,0.000023,0.003973,True,INFO
5,algo20_local_reverse_ablation,2,4,0.3,5,no_noise_reverse,0.004265,0.000023,0.003983,True,INFO
6,algo20_local_reverse_ablation,2,4,0.3,10,current_stochastic,0.004986,0.000009,0.004892,True,INFO
7,algo20_local_reverse_ablation,2,4,0.3,10,flow_ode_like,0.004238,0.000024,0.003941,True,INFO
8,algo20_local_reverse_ablation,2,4,0.3,10,no_noise_reverse,0.004239,0.000024,0.003942,True,INFO
9,algo20_local_reverse_ablation,2,4,0.3,25,current_stochastic,0.005010,0.000013,0.004880,True,INFO


## Test 7 — Gradient alignment

**Purpose:** Measure whether the witness gradient points in the same direction as the GT RMSE gradient. If the cosine is negative, no amount of sampler tuning will make pure witness PPR optimize RMSE reliably in that basin.


In [9]:
rows = []
init_modes = ('gt_noised', 'crystalformer_oracle', 'random_template')
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_gt = torch.as_tensor(chart.q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
    for t_value in CORE_T_VALUES:
        for init_mode in init_modes:
            try:
                if init_mode == 'gt_noised':
                    state, _ = sample_gt_noisy_state(case, t_value=float(t_value))
                else:
                    state, _ = sample_template_noisy_state(case, init_mode=init_mode, t_value=float(t_value))
                xi_r = torch.zeros_like(state.f, requires_grad=True)
                xi_v = torch.zeros_like(state.v, requires_grad=True)
                f_var, v_var = kldm_ppr_noise_chart(model=runner.model, f_t=state.f, v_t=state.v, xi_r=xi_r, xi_v=xi_v, t_nodes=state.t_nodes, node_index=state.node_index)
                clean = clean_prediction(replace(state, f=f_var, v=v_var))
                q_seed = sample_template_q(case, payload, mode='oracle_structure', seed=SAMPLE_SEED, dtype=state.f.dtype, device=state.f.device)
                fit = q_only_witness_fit(z_payload=map_model_to_payload_reference_chart(clean, payload), payload=payload, q_init=q_seed, q_init_mode='oracle_structure', steps=int(ALGO20_Q_ONLY_STEPS), lr=float(ALGO20_LR), grad_clip=float(ALGO20_GRAD_CLIP))
                q_star = fit['q_star']
                z_payload = map_model_to_payload_reference_chart(clean, payload)
                z_sym = chart.expand_q(q_star, device=state.f.device, dtype=state.f.dtype)
                witness_loss = witness_torus_sin_loss(z_payload, z_sym)
                gt_loss = wrapdiff(clean, case.gt_frac).square().mean()
                gw = torch.autograd.grad(witness_loss, [xi_r, xi_v], retain_graph=True)
                gg = torch.autograd.grad(gt_loss, [xi_r, xi_v], retain_graph=False)
                gw_flat = torch.cat([gw[0].reshape(-1), gw[1].reshape(-1)])
                gg_flat = torch.cat([gg[0].reshape(-1), gg[1].reshape(-1)])
                cosine = float(torch.dot(gw_flat, gg_flat) / (torch.linalg.norm(gw_flat) * torch.linalg.norm(gg_flat) + 1e-12))
                rows.append({
                    'test': 'algo20_gradient_alignment',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    't': float(t_value),
                    'init_mode': init_mode,
                    'cosine_grad_witness_vs_GT': cosine,
                    'witness_grad_norm': float(torch.linalg.norm(gw_flat).detach().item()),
                    'GT_grad_norm': float(torch.linalg.norm(gg_flat).detach().item()),
                    'PASS': True,
                    'status': 'INFO',
                })
            except Exception as exc:
                rows.append(error_row('algo20_gradient_alignment', case.graph_id, exc, 'gradient_alignment', space_group=case.requested_sg, t=float(t_value), init_mode=init_mode))

df = dataframe_result('algo20_gradient_alignment', rows)
safe_display_sorted(df, ['graph','t','init_mode'])


,test,graph,space_group,t,init_mode,cosine_grad_witness_vs_GT,witness_grad_norm,GT_grad_norm,PASS,status
0,algo20_gradient_alignment,2,4,0.1,crystalformer_oracle,0.376362,0.000015,0.000007,True,INFO
1,algo20_gradient_alignment,2,4,0.1,gt_noised,0.337513,0.000010,0.000005,True,INFO
2,algo20_gradient_alignment,2,4,0.1,random_template,-0.438371,0.000472,0.001548,True,INFO
3,algo20_gradient_alignment,2,4,0.3,crystalformer_oracle,0.017136,0.000166,0.000078,True,INFO
4,algo20_gradient_alignment,2,4,0.3,gt_noised,0.350518,0.000214,0.000064,True,INFO
5,algo20_gradient_alignment,2,4,0.3,random_template,-0.504026,0.067701,0.041604,True,INFO
6,algo20_gradient_alignment,2,4,0.5,crystalformer_oracle,0.696126,0.004466,0.000667,True,INFO
7,algo20_gradient_alignment,2,4,0.5,gt_noised,0.691756,0.004538,0.000504,True,INFO
8,algo20_gradient_alignment,2,4,0.5,random_template,-0.434252,0.364050,0.148946,True,INFO


## Test 8 — Renoise roundtrip survival

**Purpose:** Test whether a fitted q survives one renoise-denoise roundtrip at fixed time. This isolates whether the project-renoise kernel preserves or destroys the useful signal.


In [10]:
rows = []
roundtrip_modes = ('crystalformer_oracle', 'random_template')
roundtrip_t_values = CORE_T_VALUES
print(f'renoise roundtrip modes={roundtrip_modes} t={roundtrip_t_values}')
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_gt = torch.as_tensor(chart.q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
    for init_mode in roundtrip_modes:
        q_seed = sample_template_q(case, payload, mode=init_mode, seed=SAMPLE_SEED, dtype=case.gt_frac.dtype, device=case.gt_frac.device)
        f0_model, _ = build_template_f0_model(case, payload, q0=q_seed)
        for t_value in roundtrip_t_values:
            try:
                batch_view = make_single_graph_batch_view(case, ref_tensor=f0_model)
                t_graph, t_nodes = algo_times(case, float(t_value))
                set_seed(int(SAMPLE_SEED) + 440000 + int(case.graph_id) + int(round(1000 * t_value)))
                f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=f0_model, index=batch_view.batch)
                state = make_algo_state_from_raw(f=f_t, v=v_t, l=case.gt_l_feature, atom_types=case.atomic_numbers, node_index=batch_view.batch, edge_node_index=batch_view.edge_node_index, t_graph=t_graph, t_nodes=t_nodes)
                f_hat = clean_prediction(state)
                fit_hat = fit_clean_and_stats(f_hat, case, payload, q_init_mode='oracle_structure', q_init=q_seed)
                f_proj = map_payload_reference_chart_to_model_frame(fit_hat['fit']['z_proj_payload'], payload)
                red_state = renoise_clean_to_same_t(state, f_proj)
                f_again = clean_prediction(red_state)
                fit_again = fit_clean_and_stats(f_again, case, payload, q_init_mode='oracle_structure', q_init=fit_hat['fit']['q_star'])
                rows.append({
                    'test': 'algo20_renoise_roundtrip_survival',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    'init_mode': init_mode,
                    't': float(t_value),
                    'q_distance_hat_to_seed': q_distance(fit_hat['fit']['q_star'], q_seed),
                    'q_distance_again_to_hat': q_distance(fit_again['fit']['q_star'], fit_hat['fit']['q_star']),
                    'witness_before': fit_hat['witness_sin'],
                    'witness_after_roundtrip': fit_again['witness_sin'],
                    'frac_rmse_before': fit_hat['frac_rmse_to_GT'],
                    'frac_rmse_after_roundtrip': fit_again['frac_rmse_to_GT'],
                    'PASS': True,
                    'status': 'INFO',
                })
            except Exception as exc:
                rows.append(error_row('algo20_renoise_roundtrip_survival', case.graph_id, exc, 'renoise_roundtrip_survival', space_group=case.requested_sg, init_mode=init_mode, t=float(t_value)))

df = dataframe_result('algo20_renoise_roundtrip_survival', rows)
safe_display_sorted(df, ['graph','init_mode','t'])


renoise roundtrip modes=('crystalformer_oracle', 'random_template') t=(0.1, 0.3, 0.5)


,test,graph,space_group,init_mode,t,q_distance_hat_to_seed,q_distance_again_to_hat,witness_before,witness_after_roundtrip,frac_rmse_before,frac_rmse_after_roundtrip,PASS,status
0,algo20_renoise_roundtrip_survival,2,4,crystalformer_oracle,0.1,0.003801,0.001281,0.000005,0.000008,0.003871,0.004255,True,INFO
1,algo20_renoise_roundtrip_survival,2,4,crystalformer_oracle,0.3,0.004915,0.002384,0.000025,0.000048,0.005165,0.005355,True,INFO
2,algo20_renoise_roundtrip_survival,2,4,crystalformer_oracle,0.5,0.005704,0.005228,0.000085,0.000120,0.006417,0.007316,True,INFO
3,algo20_renoise_roundtrip_survival,2,4,random_template,0.1,0.013279,0.018084,0.000525,0.000398,0.197028,0.192664,True,INFO
4,algo20_renoise_roundtrip_survival,2,4,random_template,0.3,0.061885,0.063565,0.008319,0.003630,0.181826,0.182983,True,INFO
5,algo20_renoise_roundtrip_survival,2,4,random_template,0.5,0.090680,0.059797,0.047327,0.020561,0.182176,0.169349,True,INFO


## Test 9 — q multistart ambiguity

**Purpose:** Check whether q-fitting is unique or whether several q solutions achieve nearly the same witness. This helps separate real basin drift from q-optimizer ambiguity.


In [11]:
rows = []
multistart_t_values = CORE_T_VALUES
num_random_starts = 4
print(f'q multistart t={multistart_t_values} random_starts={num_random_starts}')
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_gt = torch.as_tensor(chart.q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
    q_seed = sample_template_q(case, payload, mode='crystalformer_oracle', seed=SAMPLE_SEED, dtype=case.gt_frac.dtype, device=case.gt_frac.device)
    for t_value in multistart_t_values:
        try:
            state, _ = sample_gt_noisy_state(case, t_value=float(t_value))
            clean = clean_prediction(state)
            z_payload = map_model_to_payload_reference_chart(clean, payload)
            starts = [('q_GT', q_gt), ('q_seed', q_seed), ('q_live', q_seed)]
            for i in range(num_random_starts):
                set_seed(int(SAMPLE_SEED) + 550000 + 10 * int(case.graph_id) + i)
                starts.append((f'random_{i+1}', torch.rand_like(q_gt)))
            sols = []
            for name, q0 in starts:
                fit = q_only_witness_fit(z_payload=z_payload, payload=payload, q_init=q0, q_init_mode='oracle_structure', steps=int(ALGO20_Q_ONLY_STEPS), lr=float(ALGO20_LR), grad_clip=float(ALGO20_GRAD_CLIP))
                sols.append((name, fit['witness_sin'], fit['q_star']))
            sols = sorted(sols, key=lambda x: x[1])
            best_name, best_wit, best_q = sols[0]
            second_name, second_wit, second_q = sols[1]
            rows.append({
                'test': 'algo20_q_multistart_ambiguity',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                't': float(t_value),
                'best_start': best_name,
                'second_best_start': second_name,
                'best_witness': float(best_wit),
                'second_best_witness': float(second_wit),
                'q_distance_between_solutions': q_distance(best_q, second_q),
                'q_distance_to_GT': q_distance(best_q, q_gt),
                'PASS': True,
                'status': 'INFO',
            })
        except Exception as exc:
            rows.append(error_row('algo20_q_multistart_ambiguity', case.graph_id, exc, 'q_multistart_ambiguity', space_group=case.requested_sg, t=float(t_value)))

df = dataframe_result('algo20_q_multistart_ambiguity', rows)
safe_display_sorted(df, ['graph','t'])


q multistart t=(0.1, 0.3, 0.5) random_starts=4


,test,graph,space_group,t,best_start,second_best_start,best_witness,second_best_witness,q_distance_between_solutions,q_distance_to_GT,PASS,status
0,algo20_q_multistart_ambiguity,2,4,0.1,q_GT,q_seed,0.000007,0.000007,0.000000e+00,0.002765,True,INFO
1,algo20_q_multistart_ambiguity,2,4,0.3,random_3,q_GT,0.000036,0.000036,5.252990e-07,0.004750,True,INFO
2,algo20_q_multistart_ambiguity,2,4,0.5,random_3,random_2,0.000196,0.000196,3.987250e-06,0.006244,True,INFO


## Test 10 — Frame / payload identity test

**Purpose:** Re-verify the deterministic symmetry backend: q to payload, payload to model, back to payload, then q-fit again. This should be essentially exact if the bridge is healthy.


In [12]:
rows = []
for case in GRAPH_CASES:
    try:
        payload = build_oracle_payload(case)
        chart = algo19_backend._get_wyckoff_dof_chart(payload)
        q_gt = torch.as_tensor(chart.q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
        z_payload = chart.expand_q(q_gt, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
        f_model = map_payload_reference_chart_to_model_frame(z_payload, payload)
        z_back = map_model_to_payload_reference_chart(f_model, payload)
        fit_back = q_only_witness_fit(z_payload=z_back, payload=payload, q_init=q_gt, q_init_mode='oracle_structure', steps=int(ALGO20_Q_ONLY_STEPS), lr=float(ALGO20_LR), grad_clip=float(ALGO20_GRAD_CLIP))
        species_ok = list(map(int, payload.expanded_atomic_numbers)) == list(map(int, payload.expanded_atomic_numbers))
        rows.append({
            'test': 'algo20_frame_payload_identity',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'payload_roundtrip_rmse': float(torus_rmse(z_payload, z_payload).detach().item()),
            'model_payload_roundtrip_rmse': float(torus_rmse(z_payload, z_back).detach().item()),
            'q_recovery_error': q_distance(fit_back['q_star'], q_gt),
            'witness_sin': float(fit_back['witness_sin']),
            'species_order_ok': bool(species_ok),
            'PASS': True,
            'status': 'INFO',
        })
    except Exception as exc:
        rows.append(error_row('algo20_frame_payload_identity', case.graph_id, exc, 'frame_payload_identity', space_group=case.requested_sg))

df = dataframe_result('algo20_frame_payload_identity', rows)
safe_display_sorted(df, ['graph'])


,test,graph,space_group,payload_roundtrip_rmse,model_payload_roundtrip_rmse,q_recovery_error,witness_sin,species_order_ok,PASS,status
0,algo20_frame_payload_identity,2,4,0.0,9.618658e-09,0.0,9.131220e-16,True,True,INFO


## Test 11 — Lattice sensitivity without lattice projection

**Purpose:** Hold the same `f_t, v_t` fixed and only swap the lattice passed into `D_f`. This tells us how much lattice conditioning alone changes the denoiser output.


In [13]:
rows = []
lattice_t_values = CORE_T_VALUES
print(f'lattice sensitivity t={lattice_t_values}')
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for t_value in lattice_t_values:
        try:
            state, _ = sample_gt_noisy_state(case, t_value=float(t_value))
            l_gt = case.gt_l_feature.detach().clone()
            l_frozen_seed = case.gt_l_feature.detach().clone()
            l_native = case.gt_l_feature.detach().clone()
            t_l = torch.as_tensor([float(t_value)], device=l_gt.device, dtype=l_gt.dtype)
            l_consistent_noised, _ = runner.model.diffusion_l.forward_sample(t=t_l, x0=l_gt.reshape(1,-1), num_atoms=torch.tensor([int(case.gt_frac.shape[0])], device=l_gt.device, dtype=torch.long))
            l_random_wrong = runner.model.diffusion_l.sample_prior(x_like=l_gt.reshape(1,-1), num_atoms=torch.tensor([int(case.gt_frac.shape[0])], device=l_gt.device, dtype=torch.long))
            variants = {
                'GT': l_gt,
                'frozen_seed': l_frozen_seed,
                'native_current': l_native,
                'consistently_noised': l_consistent_noised.reshape(-1),
                'random_wrong': l_random_wrong.reshape(-1),
            }
            for l_mode, lattice in variants.items():
                st = replace(state, l=lattice.detach().clone().reshape(-1))
                clean = clean_prediction(st)
                stats = fit_clean_and_stats(clean, case, payload, q_init_mode='oracle_structure')
                rows.append({
                    'test': 'algo20_lattice_sensitivity_fixed_state',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    't': float(t_value),
                    'l_mode': l_mode,
                    'witness_sin': stats['witness_sin'],
                    'frac_rmse_to_GT': stats['frac_rmse_to_GT'],
                    'q_distance_to_GT': stats['q_distance_to_GT'],
                    'valid': stats['valid'],
                    'PASS': True,
                    'status': 'INFO',
                })
        except Exception as exc:
            rows.append(error_row('algo20_lattice_sensitivity_fixed_state', case.graph_id, exc, 'lattice_sensitivity_fixed_state', space_group=case.requested_sg, t=float(t_value)))

df = dataframe_result('algo20_lattice_sensitivity_fixed_state', rows)
safe_display_sorted(df, ['graph','t','l_mode'])


lattice sensitivity t=(0.1, 0.3, 0.5)


,test,graph,space_group,t,l_mode,witness_sin,frac_rmse_to_GT,q_distance_to_GT,valid,PASS,status
0,algo20_lattice_sensitivity_fixed_state,2,4,0.1,GT,0.000007,0.002882,0.002765,True,True,INFO
1,algo20_lattice_sensitivity_fixed_state,2,4,0.1,consistently_noised,0.000006,0.002843,0.002728,True,True,INFO
2,algo20_lattice_sensitivity_fixed_state,2,4,0.1,frozen_seed,0.000007,0.002882,0.002765,True,True,INFO
3,algo20_lattice_sensitivity_fixed_state,2,4,0.1,native_current,0.000007,0.002882,0.002765,True,True,INFO
4,algo20_lattice_sensitivity_fixed_state,2,4,0.1,random_wrong,0.000007,0.003004,0.002881,True,True,INFO
5,algo20_lattice_sensitivity_fixed_state,2,4,0.3,GT,0.000036,0.005125,0.004750,True,True,INFO
6,algo20_lattice_sensitivity_fixed_state,2,4,0.3,consistently_noised,0.000030,0.004993,0.004682,True,True,INFO
7,algo20_lattice_sensitivity_fixed_state,2,4,0.3,frozen_seed,0.000036,0.005125,0.004750,True,True,INFO
8,algo20_lattice_sensitivity_fixed_state,2,4,0.3,native_current,0.000036,0.005125,0.004750,True,True,INFO
9,algo20_lattice_sensitivity_fixed_state,2,4,0.3,random_wrong,0.000027,0.004964,0.004680,True,True,INFO


## Test 12 — Projection frequency without full sampling

**Purpose:** Apply a few short project-renoise loops at one fixed time, optionally with tiny reverse drift between them. This is a cheap stand-in for 'project every 10 vs 25 steps'.


In [14]:
rows = []
K_values = (1, 2, 5, 10)
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for t_value in (0.3, 0.5):
        for K in K_values:
            try:
                state, _ = sample_gt_noisy_state(case, t_value=float(t_value))
                curr = state
                q_live = None
                for _ in range(int(K)):
                    kernel = ppr_kernel_q_witness(
                        state=curr,
                        payload=payload,
                        model=runner.model,
                        config=config20(M=1, lambda0=0.3, q_init_mode='oracle_structure', proj_steps=int(PROJ_STEPS)),
                        q_init=q_live,
                    )
                    curr = kernel.state
                    q_live = kernel.q_star
                    curr = no_noise_reverse_step(curr, dt=max(float(t_value) / max(int(K), 1) * 0.1, 1e-6))
                clean = clean_prediction(curr)
                stats = fit_clean_and_stats(clean, case, payload, q_init_mode='oracle_structure', q_init=q_live)
                rows.append({
                    'test': 'algo20_projection_frequency_short_loop',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    't': float(t_value),
                    'K': int(K),
                    'witness_sin': stats['witness_sin'],
                    'frac_rmse': stats['frac_rmse_to_GT'],
                    'q_distance_to_GT': stats['q_distance_to_GT'],
                    'PASS': True,
                    'status': 'INFO',
                })
            except Exception as exc:
                rows.append(error_row('algo20_projection_frequency_short_loop', case.graph_id, exc, 'projection_frequency_short_loop', space_group=case.requested_sg, t=float(t_value), K=int(K)))

df = dataframe_result('algo20_projection_frequency_short_loop', rows)
safe_display_sorted(df, ['graph','t','K'])


,test,graph,space_group,t,K,witness_sin,frac_rmse,q_distance_to_GT,PASS,status
0,algo20_projection_frequency_short_loop,2,4,0.3,1,0.000015,0.004646,0.004475,True,INFO
1,algo20_projection_frequency_short_loop,2,4,0.3,2,0.000025,0.004940,0.004677,True,INFO
2,algo20_projection_frequency_short_loop,2,4,0.3,5,0.000029,0.005614,0.005348,True,INFO
3,algo20_projection_frequency_short_loop,2,4,0.3,10,0.000029,0.005575,0.005305,True,INFO
4,algo20_projection_frequency_short_loop,2,4,0.5,1,0.000070,0.006017,0.005396,True,INFO
5,algo20_projection_frequency_short_loop,2,4,0.5,2,0.000099,0.008033,0.007384,True,INFO
6,algo20_projection_frequency_short_loop,2,4,0.5,5,0.000315,0.008552,0.006420,True,INFO
7,algo20_projection_frequency_short_loop,2,4,0.5,10,0.000111,0.007068,0.006220,True,INFO


## Test 13 — Hard / soft / mixed anchor ablation

**Purpose:** Compare the denoiser anchor, the hard template anchor, and simple mixed anchors. This tells us whether the current soft anchor is leaving too much symmetry error on the table.


In [ ]:
rows = []
anchor_settings = [
    ('soft', None),
    ('hard', None),
    ('mixed_0.25', 0.25),
    ('mixed_0.5', 0.5),
    ('mixed_0.75', 0.75),
]

for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for t_value in CORE_T_VALUES:
        try:
            state, _ = sample_gt_noisy_state(case, t_value=float(t_value))
            project = ppr_project_step_q_witness(
                state=state,
                payload=payload,
                model=runner.model,
                config=config20(
                    M=1,
                    lambda0=0.3,
                    q_init_mode='oracle_structure',
                    proj_steps=int(PROJ_STEPS),
                ),
                q_init=None,
            )

            soft_anchor = project.f0_star.detach().clone()
            hard_anchor = map_payload_reference_chart_to_model_frame(
                project.z_proj_payload,
                payload,
            )

            for anchor_mode, alpha in anchor_settings:
                if anchor_mode == 'soft':
                    anchor = soft_anchor
                elif anchor_mode == 'hard':
                    anchor = hard_anchor
                else:
                    a = float(alpha)
                    anchor = wrap01((1.0 - a) * soft_anchor + a * hard_anchor)

                anchor_stats = fit_clean_and_stats(
                    anchor,
                    case,
                    payload,
                    q_init_mode='oracle_structure',
                    q_init=project.q_star,
                )

                red_state = renoise_clean_to_same_t(state, anchor)
                red_clean = clean_prediction(red_state)
                red_stats = fit_clean_and_stats(
                    red_clean,
                    case,
                    payload,
                    q_init_mode='oracle_structure',
                    q_init=project.q_star,
                )

                rows.append({
                    'test': 'algo20_anchor_ablation',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    't': float(t_value),
                    'anchor_mode': anchor_mode,
                    'witness_anchor': anchor_stats['witness_sin'],
                    'frac_rmse_anchor': anchor_stats['frac_rmse_to_GT'],
                    'witness_after_redenoise': red_stats['witness_sin'],
                    'frac_rmse_after_redenoise': red_stats['frac_rmse_to_GT'],
                    'q_distance_to_GT': red_stats['q_distance_to_GT'],
                    'PASS': True,
                    'status': 'INFO',
                })

        except Exception as exc:
            rows.append(error_row(
                'algo20_anchor_ablation',
                case.graph_id,
                exc,
                'anchor_ablation',
                space_group=case.requested_sg,
                t=float(t_value),
            ))

df = dataframe_result('algo20_anchor_ablation', rows)
safe_display_sorted(df, ['graph', 't', 'anchor_mode'])


,test,graph,PASS,status,error_type,error_message,traceback_tail,failure_category,space_group,t,anchor_mode
0,algo20_anchor_ablation,2,False,ERROR,AttributeError,'Algorithm20KernelResult' object has no attrib...,AttributeError: 'Algorithm20KernelResult' obje...,anchor_ablation,4,0.1,NaN
1,algo20_anchor_ablation,2,False,ERROR,AttributeError,'Algorithm20KernelResult' object has no attrib...,AttributeError: 'Algorithm20KernelResult' obje...,anchor_ablation,4,0.3,NaN
2,algo20_anchor_ablation,2,False,ERROR,AttributeError,'Algorithm20KernelResult' object has no attrib...,AttributeError: 'Algorithm20KernelResult' obje...,anchor_ablation,4,0.5,NaN


## Notes

These diagnostics are intentionally lightweight. They focus on fixed-time slices, short local segments, small projection budgets, and one or a few graphs so that we can separate denoiser, objective, renoise, and sampler effects without paying for full chains.


## Test 14 — Post-renoise acceptance audit

**Purpose:** Audit whether we would accept the same proposal if acceptance were based on the post-renoise state instead of the projected anchor. This is a lightweight way to test the acceptance rule before changing the sampler.


In [ ]:

rows = []
accept_t_values = CORE_T_VALUES
print(f'post-renoise acceptance audit t={accept_t_values}')
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for t_value in accept_t_values:
        try:
            state, _ = sample_gt_noisy_state(case, t_value=float(t_value))
            clean_before = clean_prediction(state)
            before_stats = fit_clean_and_stats(clean_before, case, payload, q_init_mode='oracle_structure')
            project = ppr_project_step_q_witness(
                state=state,
                payload=payload,
                model=runner.model,
                config=config20(M=1, lambda0=0.3, q_init_mode='oracle_structure', proj_steps=int(PROJ_STEPS)),
                q_init=None,
            )
            anchor = project.f0_star.detach().clone()
            anchor_stats = fit_clean_and_stats(anchor, case, payload, q_init_mode='oracle_structure', q_init=project.q_star)
            red_state = renoise_clean_to_same_t(state, anchor)
            red_clean = clean_prediction(red_state)
            red_stats = fit_clean_and_stats(red_clean, case, payload, q_init_mode='oracle_structure', q_init=project.q_star)
            rows.append({
                'test': 'algo20_post_renoise_acceptance_audit',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                't': float(t_value),
                'c_before_witness': before_stats['witness_sin'],
                'c_after_project_anchor': anchor_stats['witness_sin'],
                'c_after_redenoise': red_stats['witness_sin'],
                'accept_on_anchor': bool(anchor_stats['witness_sin'] < before_stats['witness_sin']),
                'accept_on_redenoise': bool(red_stats['witness_sin'] < before_stats['witness_sin']),
                'frac_rmse_before': before_stats['frac_rmse_to_GT'],
                'frac_rmse_after_redenoise': red_stats['frac_rmse_to_GT'],
                'PASS': True,
                'status': 'INFO',
            })
        except Exception as exc:
            rows.append(error_row('algo20_post_renoise_acceptance_audit', case.graph_id, exc, 'post_renoise_acceptance_audit', space_group=case.requested_sg, t=float(t_value)))

df = dataframe_result('algo20_post_renoise_acceptance_audit', rows)
safe_display_sorted(df, ['graph','t'])


## Test 15 — q-prior strength sweep

**Purpose:** Keep the same witness-plus-q-prior architecture, but sweep a few lightweight q-prior strengths. This tells us whether the seed-region idea is promising before we commit to redesigning the main sampler.


In [ ]:

rows = []
qprior_weights = (0.0, 0.3, 1.0, 3.0)
print(f'q-prior sweep t={CORE_T_VALUES} gamma={qprior_weights}')
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for t_value in CORE_T_VALUES:
        state, _ = sample_gt_noisy_state(case, t_value=float(t_value))
        for gamma_q in qprior_weights:
            try:
                out = objective_project_step(
                    case,
                    state,
                    payload,
                    objective='witness_qprior',
                    lambda0=0.3,
                    proj_steps=int(PROJ_STEPS),
                    q_init_mode='oracle_structure',
                    q_prior_weight=float(gamma_q),
                )
                rows.append({
                    'test': 'algo20_qprior_strength_sweep',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    't': float(t_value),
                    'q_prior_weight': float(gamma_q),
                    'witness_before': out['before']['witness_sin'],
                    'witness_after_project': out['after_project']['witness_sin'],
                    'frac_rmse_before': out['before']['frac_rmse_to_GT'],
                    'frac_rmse_after_project': out['after_project']['frac_rmse_to_GT'],
                    'frac_rmse_after_redenoise': out['after_redenoise']['frac_rmse_to_GT'],
                    'q_distance_before': out['before']['q_distance_to_GT'],
                    'q_distance_after_project': out['after_project']['q_distance_to_GT'],
                    'q_distance_after_redenoise': out['after_redenoise']['q_distance_to_GT'],
                    'PASS': True,
                    'status': 'INFO',
                })
            except Exception as exc:
                rows.append(error_row('algo20_qprior_strength_sweep', case.graph_id, exc, 'qprior_strength_sweep', space_group=case.requested_sg, t=float(t_value), q_prior_weight=float(gamma_q)))

df = dataframe_result('algo20_qprior_strength_sweep', rows)
safe_display_sorted(df, ['graph','t','q_prior_weight'])


## Test 16 — Seeded partial-noise local retention

**Purpose:** Start from a template-compatible seed noised only to a moderate `t_start`, then run a very short local reverse segment. This is a lightweight version of the 'smaller `t_start`' idea, without changing the full sampler yet.


In [ ]:

rows = []
seeded_init_modes = ('crystalformer_oracle', 'random_template')
seeded_t_starts = (0.3, 0.5, 0.7)
seeded_local_steps = (1, 5, 10)
print(f'seeded partial-noise retention init_modes={seeded_init_modes} t_start={seeded_t_starts} n_local={seeded_local_steps}')
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for init_mode in seeded_init_modes:
        for t_start in seeded_t_starts:
            for n_local_steps in seeded_local_steps:
                for sampler_mode in ('current_stochastic', 'no_noise_reverse', 'flow_ode_like'):
                    try:
                        curr, _ = sample_template_noisy_state(case, init_mode=init_mode, t_value=float(t_start))
                        dt = max(float(t_start) / max(int(n_local_steps), 1), 1e-6)
                        for _ in range(int(n_local_steps)):
                            curr = reverse_modes[sampler_mode](curr, dt=dt)
                        clean = clean_prediction(curr)
                        stats = fit_clean_and_stats(clean, case, payload, q_init_mode='oracle_structure')
                        rows.append({
                            'test': 'algo20_seeded_partial_noise_local_retention',
                            'graph': case.graph_id,
                            'space_group': case.requested_sg,
                            'init_mode': init_mode,
                            't_start': float(t_start),
                            'n_local_steps': int(n_local_steps),
                            'sampler_mode': sampler_mode,
                            'frac_rmse_to_GT': stats['frac_rmse_to_GT'],
                            'witness_sin': stats['witness_sin'],
                            'q_distance_to_GT': stats['q_distance_to_GT'],
                            'valid': stats['valid'],
                            'match': stats['match'],
                            'PASS': True,
                            'status': 'INFO',
                        })
                    except Exception as exc:
                        rows.append(error_row('algo20_seeded_partial_noise_local_retention', case.graph_id, exc, 'seeded_partial_noise_local_retention', space_group=case.requested_sg, init_mode=init_mode, t_start=float(t_start), n_local_steps=int(n_local_steps), sampler_mode=sampler_mode))

df = dataframe_result('algo20_seeded_partial_noise_local_retention', rows)
safe_display_sorted(df, ['graph','init_mode','t_start','n_local_steps','sampler_mode'])
